# Update ISIN-IBES Ticker Mapping Table

This notebook rebuilds `isin/ibes_ticker.pq` from three primary sources:

1. **Compustat NA** (`d_na/security/security`) — `isin` + `ibtic`
2. **Compustat Global** (`d_global/security/g_security`) — `isin` + `ibtic`
3. **LSEG Datastream** (`ds_eq/reference_tables/wrds_ds_names`) — `ISIN` + `IBESTicker`
4. **IBES CUSIP Bridge** (NEW) — IBES `recdid`/`recdidsum` TICKER-CUSIP → CUSIP-ISIN

Each step includes row counts and quality checks.

In [20]:
import pandas as pd
import numpy as np
from datetime import date

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

TODAY = pd.to_datetime(date.today())
print(f'Run date: {TODAY.strftime("%Y-%m-%d")}')

Run date: 2026-02-24


## Step 0: Load Existing Mapping (Baseline)

In [21]:
old = pd.read_parquet('../isin/ibes_ticker.pq')
print(f'Existing mapping: {old.shape}')
print(f'  Unique ISINs:        {old["isin"].nunique():,}')
print(f'  Unique IBES tickers: {old["ibes_ticker"].nunique():,}')
print(f'  Date range:          {old["update"].min()} ~ {old["update"].max()}')
print()
print('Source breakdown:')
print(old['source'].value_counts().to_string())
print()
old.head()

Existing mapping: (104850, 4)
  Unique ISINs:        103,468
  Unique IBES tickers: 79,016
  Date range:          2022-06-16 00:00:00 ~ 2022-06-17 00:00:00

Source breakdown:
source
Refinitiv Datastream    40305
Compustat Global        39863
Compustat NA            24682



,isin,ibes_ticker,source,update
0,US8361001071,ATSPU,Refinitiv Datastream,2022-06-17
26977,US69207P3082,SYBD,Refinitiv Datastream,2022-06-17
26970,US1584832067,CPHT,Refinitiv Datastream,2022-06-17
26971,US9180761002,UTSI,Refinitiv Datastream,2022-06-17
26972,US3379301019,FLGS,Refinitiv Datastream,2022-06-17


## Step 1: Compustat NA

In [22]:
na_raw = pd.read_parquet('Z:/dataset/Compustat/d_na/security/security', 
                         columns=['isin', 'ibtic'])
print(f'Compustat NA raw: {na_raw.shape}')
print(f'  Has isin:  {na_raw["isin"].notna().sum():,}')
print(f'  Has ibtic: {na_raw["ibtic"].notna().sum():,}')

na = (na_raw
      .dropna(subset=['isin', 'ibtic'])
      .rename(columns={'ibtic': 'ibes_ticker'})
      .assign(ibes_ticker=lambda x: x['ibes_ticker'].str.upper())
      [['isin', 'ibes_ticker']]
      .drop_duplicates())

print(f'\nAfter cleaning: {na.shape}')
print(f'  Unique ISINs:        {na["isin"].nunique():,}')
print(f'  Unique IBES tickers: {na["ibes_ticker"].nunique():,}')
na.head()

Compustat NA raw: (70153, 2)
  Has isin:  54,623
  Has ibtic: 28,658

After cleaning: (24805, 2)
  Unique ISINs:        24,342
  Unique IBES tickers: 24,805


,isin,ibes_ticker
1,US0001651001,AMFD
3,US0003541002,ANTQ
4,US0003611052,AIR
9,US0007811047,ABSI
11,US0008723092,ACSE


## Step 2: Compustat Global

In [23]:
gl_raw = pd.read_parquet('Z:/dataset/Compustat/d_global/security/g_security', 
                         columns=['isin', 'ibtic'])
print(f'Compustat Global raw: {gl_raw.shape}')
print(f'  Has isin:  {gl_raw["isin"].notna().sum():,}')
print(f'  Has ibtic: {gl_raw["ibtic"].notna().sum():,}')

gl = (gl_raw
      .dropna(subset=['isin', 'ibtic'])
      .rename(columns={'ibtic': 'ibes_ticker'})
      .assign(ibes_ticker=lambda x: x['ibes_ticker'].str.upper())
      [['isin', 'ibes_ticker']]
      .drop_duplicates())

print(f'\nAfter cleaning: {gl.shape}')
print(f'  Unique ISINs:        {gl["isin"].nunique():,}')
print(f'  Unique IBES tickers: {gl["ibes_ticker"].nunique():,}')
gl.head()

Compustat Global raw: (99372, 2)
  Has isin:  78,853
  Has ibtic: 41,454

After cleaning: (39829, 2)
  Unique ISINs:        39,829
  Unique IBES tickers: 39,826


,isin,ibes_ticker
0,NL0000334118,@4M
5,IL0006320183,@A3I
6,BMG6359F1370,@66K
9,PH0492493038,@AT
10,PHY0434M1265,@YZV


## Step 3: LSEG Datastream

In [24]:
ds_raw = pd.read_parquet('Z:/dataset/LSEG/ds_eq/reference_tables/wrds_ds_names', 
                         columns=['ISIN', 'IBESTicker'])
print(f'LSEG Datastream raw: {ds_raw.shape}')
print(f'  Has ISIN:        {ds_raw["ISIN"].notna().sum():,}')
print(f'  Has IBESTicker:  {ds_raw["IBESTicker"].notna().sum():,}')

ds = (ds_raw
      .dropna(subset=['ISIN', 'IBESTicker'])
      .rename(columns={'ISIN': 'isin', 'IBESTicker': 'ibes_ticker'})
      .assign(ibes_ticker=lambda x: x['ibes_ticker'].str.replace('@:', '', regex=False).str.upper())
      [['isin', 'ibes_ticker']]
      .drop_duplicates())

print(f'\nAfter cleaning: {ds.shape}')
print(f'  Unique ISINs:        {ds["isin"].nunique():,}')
print(f'  Unique IBES tickers: {ds["ibes_ticker"].nunique():,}')
ds.head()

LSEG Datastream raw: (210575, 2)
  Has ISIN:        210,575
  Has IBESTicker:  101,598

After cleaning: (99329, 2)
  Unique ISINs:        99,180
  Unique IBES tickers: 74,259


,isin,ibes_ticker
0,MXP5874P1098,@WLQ
1,MX01IN4J0002,@WLQ
2,BRAVILACNOR1,@7AE
6,BRBDLLACNPR8,@7BL
8,PEP771431009,@00


## Step 4: IBES CUSIP Bridge (NEW pathway)

Use IBES's own identification tables (`recdid`, `recdidsum`) which contain TICKER-CUSIP pairs,
then bridge to ISIN via the existing `cusip.pq` mapping table.

This picks up IBES tickers that neither Compustat nor Datastream cross-reference.

In [25]:
# Load all IBES id tables with TICKER-CUSIP
ibes_sources = [
    'Z:/dataset/IBES/ibes/id',
    'Z:/dataset/IBES/ibes/idsum',
    'Z:/dataset/IBES/ibes/recdid',
    'Z:/dataset/IBES/ibes/recdidsum',
]

ibes_tc = []
for src in ibes_sources:
    tmp = pd.read_parquet(src, columns=['TICKER', 'CUSIP'])
    tmp = tmp.dropna(subset=['CUSIP']).drop_duplicates()
    print(f'{src.split("/")[-2]:>12}: {tmp.shape[0]:>7,} pairs, {tmp["TICKER"].nunique():>6,} tickers')
    ibes_tc.append(tmp)

ibes_tc = pd.concat(ibes_tc).drop_duplicates()
print(f'{"combined":>12}: {ibes_tc.shape[0]:>7,} pairs, {ibes_tc["TICKER"].nunique():>6,} tickers, {ibes_tc["CUSIP"].nunique():>6,} CUSIPs')

        ibes: 120,157 pairs, 85,511 tickers
        ibes: 106,967 pairs, 76,412 tickers
        ibes: 135,571 pairs, 93,963 tickers
        ibes: 133,000 pairs, 93,958 tickers
    combined: 136,128 pairs, 93,972 tickers, 134,999 CUSIPs


In [26]:
# Load ISIN-CUSIP mapping, truncate to 8-char CUSIP for matching
cusip_map = pd.read_parquet('../isin/cusip.pq', columns=['isin', 'cusip'])
cusip_map['cusip8'] = cusip_map['cusip'].str[:8]
cusip_isin = cusip_map[['isin', 'cusip8']].drop_duplicates()
print(f'CUSIP-ISIN bridge: {cusip_isin.shape[0]:,} pairs ({cusip_isin["cusip8"].nunique():,} unique CUSIP8)')

# Match
bridge = (ibes_tc
          .rename(columns={'CUSIP': 'cusip8', 'TICKER': 'ibes_ticker'})
          .merge(cusip_isin, on='cusip8', how='inner')
          [['isin', 'ibes_ticker']]
          .drop_duplicates())

print(f'\nCUSIP bridge result: {bridge.shape}')
print(f'  Unique ISINs:        {bridge["isin"].nunique():,}')
print(f'  Unique IBES tickers: {bridge["ibes_ticker"].nunique():,}')
bridge.head()

CUSIP-ISIN bridge: 614,767 pairs (614,103 unique CUSIP8)

CUSIP bridge result: (29002, 2)
  Unique ISINs:        28,732
  Unique IBES tickers: 27,596


,isin,ibes_ticker
0,US87482X1019,0000
1,US2687851020,0001
2,CA6651591092,0003
3,US02504D1081,0004
4,US1416331072,000R


## Step 5: Merge All Sources

Priority order: Compustat NA > Compustat Global > Datastream > CUSIP Bridge.

When the same `(isin, ibes_ticker)` pair appears in multiple sources, we keep the first source encountered.

In [27]:
na_out = na.copy(); na_out['source'] = 'Compustat NA'
gl_out = gl.copy(); gl_out['source'] = 'Compustat Global'
ds_out = ds.copy(); ds_out['source'] = 'Refinitiv Datastream'
br_out = bridge.copy(); br_out['source'] = 'IBES-CUSIP'

new = pd.concat([na_out, gl_out, ds_out, br_out], ignore_index=True)
new['update'] = TODAY

print(f'Before dedup: {new.shape[0]:,}')
new.drop_duplicates(subset=['isin', 'ibes_ticker'], keep='first', inplace=True)
print(f'After dedup:  {new.shape[0]:,}')
print()
print('Source breakdown:')
print(new['source'].value_counts().to_string())
print()
print(f'Unique ISINs:        {new["isin"].nunique():,}')
print(f'Unique IBES tickers: {new["ibes_ticker"].nunique():,}')
new.head()

Before dedup: 192,965
After dedup:  107,990

Source breakdown:
source
Refinitiv Datastream    41376
Compustat Global        39829
Compustat NA            24805
IBES-CUSIP               1980

Unique ISINs:        105,896
Unique IBES tickers: 81,058


,isin,ibes_ticker,source,update
0,US0001651001,AMFD,Compustat NA,2026-02-24
1,US0003541002,ANTQ,Compustat NA,2026-02-24
2,US0003611052,AIR,Compustat NA,2026-02-24
3,US0007811047,ABSI,Compustat NA,2026-02-24
4,US0008723092,ACSE,Compustat NA,2026-02-24


## Step 6: Quality Comparison — New vs Old

In [28]:
old_pairs = set(zip(old['isin'], old['ibes_ticker']))
new_pairs = set(zip(new['isin'], new['ibes_ticker']))

only_old = old_pairs - new_pairs
only_new = new_pairs - old_pairs
both = old_pairs & new_pairs

print('=== Pair-Level Comparison ===')
print(f'Old total:    {len(old_pairs):>8,}')
print(f'New total:    {len(new_pairs):>8,}')
print(f'Overlapping:  {len(both):>8,}')
print(f'Added (new):  {len(only_new):>8,}')
print(f'Dropped (old):{len(only_old):>8,}')
print(f'Net change:   {len(new_pairs) - len(old_pairs):>+8,}')
print()

old_isins = set(old['isin'])
new_isins = set(new['isin'])
print('=== ISIN-Level Comparison ===')
print(f'Old ISINs:    {len(old_isins):>8,}')
print(f'New ISINs:    {len(new_isins):>8,}')
print(f'Added ISINs:  {len(new_isins - old_isins):>8,}')
print(f'Dropped ISINs:{len(old_isins - new_isins):>8,}')
print()

old_tickers = set(old['ibes_ticker'])
new_tickers = set(new['ibes_ticker'])
print('=== Ticker-Level Comparison ===')
print(f'Old tickers:    {len(old_tickers):>8,}')
print(f'New tickers:    {len(new_tickers):>8,}')
print(f'Added tickers:  {len(new_tickers - old_tickers):>8,}')
print(f'Dropped tickers:{len(old_tickers - new_tickers):>8,}')

=== Pair-Level Comparison ===
Old total:     104,850
New total:     107,990
Overlapping:   104,131
Added (new):     3,859
Dropped (old):     719
Net change:     +3,140

=== ISIN-Level Comparison ===
Old ISINs:     103,468
New ISINs:     105,896
Added ISINs:     3,074
Dropped ISINs:     646

=== Ticker-Level Comparison ===
Old tickers:      79,016
New tickers:      81,059
Added tickers:     2,078
Dropped tickers:      35


### 6a: Inspect Dropped Pairs

Pairs that were in the old table but not in the new. These could be:
- Securities that were delisted / removed from source databases
- CUSIP/ISIN changes
- Data corrections by the source

In [29]:
if only_old:
    dropped = old.loc[old.apply(lambda r: (r['isin'], r['ibes_ticker']) in only_old, axis=1)].copy()
    print(f'Dropped pairs: {dropped.shape[0]}')
    print()
    print('By source:')
    print(dropped['source'].value_counts().to_string())
    print()

    # Are these ISINs still in the new table (just with different tickers)?
    dropped_isins = set(dropped['isin'])
    still_mapped = dropped_isins & new_isins
    fully_lost = dropped_isins - new_isins
    print(f'Dropped ISINs still mapped (different ticker): {len(still_mapped)}')
    print(f'Dropped ISINs fully lost:                      {len(fully_lost)}')
    print()
    print('Sample dropped pairs:')
    dropped.head(20)
else:
    print('No pairs were dropped.')

Dropped pairs: 719

By source:
source
Refinitiv Datastream    646
Compustat Global         50
Compustat NA             23

Dropped ISINs still mapped (different ticker): 72
Dropped ISINs fully lost:                      646

Sample dropped pairs:


,isin,ibes_ticker,source,update
29367,PLPEPES00018,@9P1,Refinitiv Datastream,2022-06-17
28569,US8227031049,@SHE,Refinitiv Datastream,2022-06-17
23322,CA8729481124,TWR1,Refinitiv Datastream,2022-06-17
104325,TH0670A10Y11,@4WU,Refinitiv Datastream,2022-06-17
104318,TH0665010Y19,@4NL,Refinitiv Datastream,2022-06-17
104319,TH0666010010,@4SK,Refinitiv Datastream,2022-06-17
104322,TH0668010018,@4TV,Refinitiv Datastream,2022-06-17
104323,TH0670010Y19,@4WU,Refinitiv Datastream,2022-06-17
104324,TH0670010Z18,@4WU,Refinitiv Datastream,2022-06-17
104326,TH0669010Z10,@4BG,Refinitiv Datastream,2022-06-17


### 6b: Inspect Added Pairs

In [30]:
if only_new:
    added = new.loc[new.apply(lambda r: (r['isin'], r['ibes_ticker']) in only_new, axis=1)].copy()
    print(f'Added pairs: {added.shape[0]}')
    print()
    print('By source:')
    print(added['source'].value_counts().to_string())
    print()

    # Are these genuinely new ISINs or existing ISINs gaining new tickers?
    added_isins = set(added['isin'])
    brand_new_isins = added_isins - old_isins
    existing_isins_new_tickers = added_isins & old_isins
    print(f'Brand new ISINs:                     {len(brand_new_isins)}')
    print(f'Existing ISINs gaining new tickers:   {len(existing_isins_new_tickers)}')
    print()
    print('Sample added pairs:')
    added.head(20)
else:
    print('No new pairs were added.')

Added pairs: 3859

By source:
source
IBES-CUSIP              1949
Refinitiv Datastream    1011
Compustat NA             627
Compustat Global         272

Brand new ISINs:                     3074
Existing ISINs gaining new tickers:   714

Sample added pairs:


,isin,ibes_ticker,source,update
43,US05587G2030,ADGE,Compustat NA,2026-02-24
460,CA11271J1075,BL1,Compustat NA,2026-02-24
461,CA11271J1075,BL2,Compustat NA,2026-02-24
563,CA13646K1084,CP,Compustat NA,2026-02-24
564,CA13646K1084,CPR1,Compustat NA,2026-02-24
859,US2244081046,CR,Compustat NA,2026-02-24
1054,US2681575005,DYNT,Compustat NA,2026-02-24
1228,US1280583022,PHSM,Compustat NA,2026-02-24
1382,US78473E1038,SPW,Compustat NA,2026-02-24
1409,GB00BN7SWP63,GLAX,Compustat NA,2026-02-24


### 6c: Source Migration

For overlapping pairs, did the attributed source change?

In [31]:
# Build lookup dicts for source comparison
old_src = old.set_index(['isin', 'ibes_ticker'])['source'].to_dict()
new_src = new.set_index(['isin', 'ibes_ticker'])['source'].to_dict()

migrations = []
for pair in both:
    os = old_src.get(pair, '?')
    ns = new_src.get(pair, '?')
    if os != ns:
        migrations.append({'isin': pair[0], 'ibes_ticker': pair[1], 'old_source': os, 'new_source': ns})

print(f'Pairs with source change: {len(migrations)} / {len(both)} overlapping')
if migrations:
    mig_df = pd.DataFrame(migrations)
    print()
    print('Migration matrix (old → new):')
    print(pd.crosstab(mig_df['old_source'], mig_df['new_source']))
    print()
    mig_df.head(10)

Pairs with source change: 1022 / 104131 overlapping

Migration matrix (old → new):
new_source            Compustat Global  Compustat NA  IBES-CUSIP  Refinitiv Datastream
old_source                                                                            
Compustat Global                     0             0           0                   362
Compustat NA                         0             0          30                   487
Refinitiv Datastream               106            36           1                     0



,isin,ibes_ticker,old_source,new_source
0,US92840H2022,00PM,Compustat NA,Refinitiv Datastream
1,US5150691021,LABP,Compustat NA,Refinitiv Datastream
2,US72941H4002,CYTX,Compustat NA,Refinitiv Datastream
3,US57055L1070,GMXX,Compustat NA,Refinitiv Datastream
4,US62844N2080,MYSZ,Compustat NA,Refinitiv Datastream
5,AU0000185597,@EAF,Refinitiv Datastream,Compustat Global
6,LV0000101079,@UKI,Compustat Global,Refinitiv Datastream
7,US7574681034,RDHL,Compustat NA,Refinitiv Datastream
8,SE0000101297,@CF7,Compustat Global,Refinitiv Datastream
9,AU000000ABP9,@LK3,Compustat Global,Refinitiv Datastream


### 6d: Data Integrity Checks

In [32]:
print('=== Integrity Checks ===')
print()

# Check 1: No nulls
nulls = new.isnull().sum()
print(f'1. Null check: {"PASS" if nulls.sum() == 0 else "FAIL"}')
if nulls.sum() > 0:
    print(nulls[nulls > 0])
print()

# Check 2: No duplicates on (isin, ibes_ticker)
dups = new.duplicated(subset=['isin', 'ibes_ticker']).sum()
print(f'2. Duplicate check (isin, ibes_ticker): {"PASS" if dups == 0 else f"FAIL ({dups} dups)"}')
print()

# Check 3: ISIN format (12 chars, alphanumeric)
bad_isin = new.loc[~new['isin'].str.match(r'^[A-Z]{2}[A-Z0-9]{10}$')]
print(f'3. ISIN format check: {bad_isin.shape[0]} non-standard ISINs')
if bad_isin.shape[0] > 0:
    print('   Sample bad ISINs:', bad_isin['isin'].head(10).tolist())
print()

# Check 4: IBES ticker non-empty / reasonable
empty_ticker = new.loc[new['ibes_ticker'].str.strip() == '']
print(f'4. Empty ticker check: {"PASS" if empty_ticker.shape[0] == 0 else f"FAIL ({empty_ticker.shape[0]})"}')
print()

# Check 5: One-to-many stats
isin_per_ticker = new.groupby('ibes_ticker')['isin'].nunique()
ticker_per_isin = new.groupby('isin')['ibes_ticker'].nunique()
print(f'5. Mapping cardinality:')
print(f'   ISINs per ticker — mean: {isin_per_ticker.mean():.2f}, max: {isin_per_ticker.max()}, median: {isin_per_ticker.median():.0f}')
print(f'   Tickers per ISIN — mean: {ticker_per_isin.mean():.2f}, max: {ticker_per_isin.max()}, median: {ticker_per_isin.median():.0f}')
print()

# Flag ISINs with suspiciously many tickers
many_tickers = ticker_per_isin[ticker_per_isin > 3]
if len(many_tickers) > 0:
    print(f'   ISINs with >3 tickers: {len(many_tickers)}')
    print(many_tickers.sort_values(ascending=False).head(10))

=== Integrity Checks ===

1. Null check: FAIL
ibes_ticker    1
dtype: int64

2. Duplicate check (isin, ibes_ticker): PASS

3. ISIN format check: 0 non-standard ISINs

4. Empty ticker check: PASS

5. Mapping cardinality:
   ISINs per ticker — mean: 1.33, max: 23, median: 1
   Tickers per ISIN — mean: 1.02, max: 4, median: 1

   ISINs with >3 tickers: 2
isin
AU000000KLA9    4
US69047Q1022    4
Name: ibes_ticker, dtype: int64


### 6e: Validate Against IBES Data

Cross-check: how many tickers in our new mapping actually exist in the IBES database?

In [33]:
# Load all known IBES tickers from the id/recdid tables
ibes_known = set()
for src in ibes_sources:
    tmp = pd.read_parquet(src, columns=['TICKER'])
    ibes_known.update(tmp['TICKER'].dropna().unique())

print(f'Known IBES tickers (from IBES id tables): {len(ibes_known):,}')

new_tickers_set = set(new['ibes_ticker'])
in_ibes = new_tickers_set & ibes_known
not_in_ibes = new_tickers_set - ibes_known

print(f'Our tickers found in IBES:     {len(in_ibes):,} ({len(in_ibes)/len(new_tickers_set)*100:.1f}%)')
print(f'Our tickers NOT found in IBES: {len(not_in_ibes):,} ({len(not_in_ibes)/len(new_tickers_set)*100:.1f}%)')

if not_in_ibes:
    print(f'\nSample tickers not in IBES (may be valid but inactive):')
    sample = list(not_in_ibes)[:20]
    print(sample)

Known IBES tickers (from IBES id tables): 93,988
Our tickers found in IBES:     80,739 (99.6%)
Our tickers NOT found in IBES: 320 (0.4%)

Sample tickers not in IBES (may be valid but inactive):
['@FR2M1', 'ANBY', '@A9V', 'RAMEM1', '@SMCM1', '@5XCM1', '@YAPM1', '@K5KM1', 'AKOM1', 'LENM1', 'OASSM1', 'YRI2', '@AATE', '@IUSM4', 'BI0E', 'TPGHM1', 'JMCC', 'LILAM1', '@2S7', '@DANM1']


### 6f: Contribution of the CUSIP Bridge

How many pairs are exclusively from the CUSIP bridge (not available from any other source)?

In [34]:
direct_pairs = set(zip(
    pd.concat([na_out, gl_out, ds_out])['isin'],
    pd.concat([na_out, gl_out, ds_out])['ibes_ticker']
))
bridge_pairs = set(zip(br_out['isin'], br_out['ibes_ticker']))

bridge_exclusive = bridge_pairs - direct_pairs
bridge_overlap = bridge_pairs & direct_pairs

print(f'CUSIP bridge total pairs:     {len(bridge_pairs):,}')
print(f'CUSIP bridge exclusive pairs: {len(bridge_exclusive):,} (genuinely new)')
print(f'CUSIP bridge overlapping:     {len(bridge_overlap):,} (confirmed by other sources)')

# ISINs gained exclusively from bridge
bridge_exclusive_isins = set(p[0] for p in bridge_exclusive) - set(p[0] for p in direct_pairs)
print(f'ISINs only reachable via CUSIP bridge: {len(bridge_exclusive_isins):,}')

CUSIP bridge total pairs:     29,002
CUSIP bridge exclusive pairs: 1,980 (genuinely new)
CUSIP bridge overlapping:     27,022 (confirmed by other sources)
ISINs only reachable via CUSIP bridge: 1,272


## Step 7: Final Preview

In [35]:
# Ensure column order and dtypes match the existing file
new = new[['isin', 'ibes_ticker', 'source', 'update']].copy()
new['update'] = new['update'].astype('datetime64[us]')

print(f'Final table: {new.shape}')
print()
print('Dtypes:')
print(new.dtypes)
print()
print('Source breakdown:')
print(new['source'].value_counts().to_string())
print()
new.sample(10)

Final table: (107990, 4)

Dtypes:
isin                   object
ibes_ticker            object
source                 object
update         datetime64[us]
dtype: object

Source breakdown:
source
Refinitiv Datastream    41376
Compustat Global        39829
Compustat NA            24805
IBES-CUSIP               1980



,isin,ibes_ticker,source,update
141742,US92835A1051,GCHT,Refinitiv Datastream,2026-02-24
22678,US87164U4094,SHM,Compustat NA,2026-02-24
108967,US3633841081,HE10,Refinitiv Datastream,2026-02-24
161070,JP3830720003,@7352,Refinitiv Datastream,2026-02-24
48781,INE335C01018,@47K,Compustat Global,2026-02-24
13055,US4563141039,IDSA,Compustat NA,2026-02-24
24104,US1413371055,CARZ,Compustat NA,2026-02-24
64547,CNE100002RS7,@0189,Compustat Global,2026-02-24
2629,US7414001054,PRIA,Compustat NA,2026-02-24
54350,SE0001742743,@WU8,Compustat Global,2026-02-24


## Step 8: Save

**Review the above checks before running this cell.**

In [36]:
import time

# Fix: drop the 1 null ibes_ticker row found in integrity check
before = new.shape[0]
new = new.dropna(subset=['ibes_ticker'])
print(f'Dropped {before - new.shape[0]} null ticker row(s). Final: {new.shape[0]:,} pairs')
print()

output_path = '../isin/ibes_ticker.pq'

# Safety: backup old file
old_backup = pd.read_parquet(output_path)
old_backup.to_parquet(output_path.replace('.pq', '_backup.pq'), index=False)
print(f'Backup saved to {output_path.replace(".pq", "_backup.pq")}')

time.sleep(3)  # Pause before overwriting
new.to_parquet(output_path, index=False)
print(f'Saved to {output_path}')

# Verify
verify = pd.read_parquet(output_path)
assert verify.shape == new.shape, f'Shape mismatch: {verify.shape} vs {new.shape}'
assert (verify.columns == new.columns).all(), 'Column mismatch'
print(f'Verified: {verify.shape}')

Dropped 1 null ticker row(s). Final: 107,989 pairs

Backup saved to ../isin/ibes_ticker_backup.pq
Saved to ../isin/ibes_ticker.pq
Verified: (107989, 4)
